# RepoGuard — Phase 2: RL Agent Development, Training, Evaluation & Cloud Pipeline

> **Project:** RepoGuard (Final Year Engineering Project)  
> **Workstream:** Member 3 — RL Agent Development, Training, Evaluation & Cloud Pipeline  
> **Repository:** [https://github.com/Shreyank86/Repo-Guard](https://github.com/Shreyank86/Repo-Guard)  
> **Goal:** Production-grade 2,000-episode DQN & 1,500-episode REINFORCE training on GPU with **automatic checkpoint resume**, **persistent Drive caching & index reuse**, and **held-out test split evaluation**.

---

### Architectural Invariants Preserved:
- **Frozen LLM:** `Qwen/Qwen2.5-Coder-3B-Instruct` (4-bit NF4 quantization on CUDA via `bitsandbytes`).
- **Embedding Model:** `sentence-transformers/all-MiniLM-L6-v2`.
- **State Space:** 18-dimensional normalized state vector ($s \in [0, 1]^{18}$).
- **Action Space:** Canonical 6 actions (`SEARCH`, `INSPECT`, `VERIFY`, `ANSWER`, `ANSWER_WITH_EVIDENCE`, `ABSTAIN`).
- **Risk Parameter:** $\lambda \in [0.25, 5.0]$ (dynamic risk tolerance sampled per episode).
- **Algorithms:** Deep Q-Network (DQN) with target network & action masking, and REINFORCE with running-mean baseline.
- **Dataset Splits:** Strict boundary isolation — Train (36 cases), Validation (12 cases), Held-Out Test (12 cases).
- **Persistent Storage:** Google Drive canonical storage for vector indexes, SQLite operation cache, checkpoints, and reports.

## 1. GPU and Runtime Validation
Verifies that Google Colab has allocated an NVIDIA GPU (T4, V100, or A100) and that PyTorch can access CUDA. Fails immediately if CUDA is unavailable.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

# Strict Hard Assertion: Halt immediately if GPU is missing
assert torch.cuda.is_available(), (
    "FATAL ERROR: CUDA is not available in this runtime!\n"
    "Please switch hardware accelerator: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU (or A100)."
)

gpu_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
cuda_arch = torch.cuda.get_device_capability(0)
print(f"[OK] GPU Allocated: {gpu_name} ({total_vram_gb:.2f} GB VRAM, Compute Capability: {cuda_arch})")

## 2. Google Drive Mount and Persistent Directories
Mount Google Drive so that vector indexes, cached LLM outputs, checkpoints, and evaluation reports persist permanently across disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/RepoGuard')
DRIVE_INDEXES = DRIVE_ROOT / 'data' / 'indexes'
DRIVE_CACHE = DRIVE_ROOT / 'data' / 'cache'
DRIVE_EPISODES = DRIVE_ROOT / 'data' / 'episodes'
DRIVE_CHECKPOINTS_DQN = DRIVE_ROOT / 'checkpoints' / 'dqn'
DRIVE_CHECKPOINTS_RF = DRIVE_ROOT / 'checkpoints' / 'reinforce'
DRIVE_REPORTS = DRIVE_ROOT / 'reports'

persistent_dirs = [
    DRIVE_INDEXES,
    DRIVE_CACHE,
    DRIVE_EPISODES,
    DRIVE_CHECKPOINTS_DQN,
    DRIVE_CHECKPOINTS_RF,
    DRIVE_REPORTS,
]
for d in persistent_dirs:
    d.mkdir(parents=True, exist_ok=True)
    assert d.exists(), f"Failed to create persistent directory: {d}"

print(f"[OK] Persistent Drive workspace verified at: {DRIVE_ROOT}")

## 3. Repository Clone and Commit Recording
Clone the RepoGuard GitHub repository (or pull latest changes if already cloned) and record the exact commit hash.

In [ ]:
%cd /content
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Shreyank86/Repo-Guard.git"
repo_path = Path('/content/Repo-Guard')

if not repo_path.exists():
    print("Cloning repository from GitHub...")
    subprocess.run(["git", "clone", REPO_URL, "/content/Repo-Guard"], check=True)
else:
    print("Resetting local repository to latest origin/main...")
    subprocess.run(["git", "fetch", "origin", "main"], cwd=repo_path, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=repo_path, check=True)

%cd /content/Repo-Guard
git_commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
git_summary = subprocess.run(["git", "log", "-1", "--format=%h - %s (%ci)"], capture_output=True, text=True, check=True).stdout.strip()
print(f"[OK] Repository Active: {git_commit} ({git_summary})")


## 4. Dependency Installation
Install project requirements, `bitsandbytes` (for 4-bit NF4 quantized inference), and the `repoguard` package in editable mode.

In [ ]:
%cd /content/Repo-Guard
import os
import sys
import subprocess
from pathlib import Path

REPO_DIR = Path('/content/Repo-Guard')
os.chdir(str(REPO_DIR))

print("1. Installing dependencies from requirements.txt...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=str(REPO_DIR), check=True)

print("2. Installing bitsandbytes...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes'], cwd=str(REPO_DIR), check=True)

print("3. Installing RepoGuard in editable mode...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'setuptools', 'wheel'], cwd=str(REPO_DIR), check=False)

res = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], cwd=str(REPO_DIR))
if res.returncode != 0:
    print("  [NOTE] pip editable mode hook returned non-zero; falling back to setup.py develop...")
    res2 = subprocess.run([sys.executable, 'setup.py', 'develop', '--no-deps'], cwd=str(REPO_DIR))
    if res2.returncode != 0:
        print("  [NOTE] setup.py develop failed; creating .pth file directly in site-packages...")
        site_pkgs = [p for p in sys.path if 'site-packages' in p]
        if site_pkgs:
            pth_file = Path(site_pkgs[0]) / 'repoguard.pth'
            pth_file.write_text(str((REPO_DIR / 'src').resolve()) + '\n', encoding='utf-8')
            print(f"  [OK] Created {pth_file} -> src directory")

src_path = str((REPO_DIR / 'src').resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import repoguard
import bitsandbytes
import transformers
print(f"[OK] RepoGuard v0.1.0 successfully imported from: {repoguard.__file__}")
print("[OK] bitsandbytes and transformers ready for GPU!")


## 5. Repository and Index Preparation
Clones the 5 pinned benchmark repositories (`schedule`, `python-dotenv`, `cachetools`, `python-tabulate`, `requests`).  
- **Validation before reuse:** Checks each index in Google Drive for correct schema version (`1.1`), pinned commit hash, and embedding model (`sentence-transformers/all-MiniLM-L6-v2`).  
- **Rebuilds only when missing or invalid:** Reuses valid indexes immediately via symlink, avoiding unnecessary re-indexing on every run.

In [ ]:
%cd /content/Repo-Guard
import os
import sys
import json
import shutil
import yaml
import subprocess
from pathlib import Path
from repoguard.repository.indexer import load_index

# 1. Fetch source code for the 5 pinned repositories
subprocess.run([sys.executable, "scripts/fetch_repos.py"], check=True)

# 2. Inspect Drive indexes for schema version, commit hash, and embedding model
repos_cfg = yaml.safe_load(Path("configs/repos.yaml").read_text(encoding="utf-8"))["repositories"]
local_idx_dir = Path("/content/Repo-Guard/data/indexes")
local_idx_dir.parent.mkdir(parents=True, exist_ok=True)

def is_index_valid(index_path: Path, expected_commit: str) -> bool:
    manifest_file = index_path / "manifest.json"
    required_files = ["manifest.json", "symbols.json", "callgraph.json", "passages.jsonl", "vector_meta.json"]
    if not all((index_path / f).exists() for f in required_files):
        return False
    # Accept either passages.npy or passages.faiss vector store
    if not ((index_path / "passages.npy").exists() or (index_path / "passages.faiss").exists()):
        return False
    try:
        data = json.loads(manifest_file.read_text(encoding="utf-8"))
        schema_ok = data.get("schema_version") == "1.1"
        commit_ok = data.get("repository", {}).get("commit", "").startswith(expected_commit[:7])
        model_ok = data.get("tool_versions", {}).get("embedding_model") == "sentence-transformers/all-MiniLM-L6-v2"
        idx = load_index(index_path)
        content_ok = len(idx.symbols) > 0 and len(idx.passages) > 0
        return schema_ok and commit_ok and model_ok and content_ok
    except Exception:
        return False

all_valid = True
for r in repos_cfg:
    name, commit = r["name"], r["commit"]
    drive_idx = DRIVE_INDEXES / name
    if is_index_valid(drive_idx, commit):
        print(f"  [OK] Valid persistent index found in Drive for: {name}")
    else:
        print(f"  [MISSING/INVALID] Index needs to be built for: {name}")
        all_valid = False

if all_valid:
    print("\nAll 5 indexes valid in Google Drive! Reusing without rebuild.")
    if local_idx_dir.is_symlink():
        local_idx_dir.unlink()
    elif local_idx_dir.exists():
        shutil.rmtree(local_idx_dir, ignore_errors=True)
    os.symlink(str(DRIVE_INDEXES), str(local_idx_dir))
    print(f"Symlinked {DRIVE_INDEXES} -> {local_idx_dir}")
else:
    print("\nBuilding missing indexes and persisting them to Google Drive...")
    subprocess.run([sys.executable, "scripts/fetch_repos.py", "--index"], check=True)
    for r in repos_cfg:
        src = local_idx_dir / r["name"]
        dst = DRIVE_INDEXES / r["name"]
        if src.exists():
            if dst.exists():
                shutil.rmtree(dst, ignore_errors=True)
            shutil.copytree(src, dst)
    print("Indexes built and backed up to Google Drive successfully!")


## 6. Benchmark and Split Validation
Enforces the dataset boundary contract across all 60 benchmark cases:  
- **Train:** exactly 36 cases (`schedule`, `python-dotenv`, `cachetools`)  
- **Validation:** exactly 12 cases (`python-tabulate`)  
- **Test:** exactly 12 cases (`requests` held out)  
- Asserts zero overlap between split case IDs or question texts, and verifies facts.

In [ ]:
%cd /content/Repo-Guard
from pathlib import Path
from collections import Counter
from repoguard.evaluation.benchmark import load_benchmark

cases = load_benchmark(Path("data/questions/phase2_benchmark_v1.jsonl"))
assert len(cases) == 60, f"Expected 60 benchmark cases, got {len(cases)}"

split_counts = Counter(c.split for c in cases)
print(f"Benchmark Split Breakdown: {dict(split_counts)}")

assert split_counts["train"] == 36, f"Expected 36 train cases, got {split_counts['train']}"
assert split_counts["validation"] == 12, f"Expected 12 validation cases, got {split_counts['validation']}"
assert split_counts["test"] == 12, f"Expected 12 test cases, got {split_counts['test']}"

# Strict Isolation Checks
train_repos = {c.repository_id for c in cases if c.split == 'train'}
val_repos = {c.repository_id for c in cases if c.split == 'validation'}
test_repos = {c.repository_id for c in cases if c.split == 'test'}

assert train_repos == {"schedule", "python-dotenv", "cachetools"}, f"Unexpected train repos: {train_repos}"
assert val_repos == {"python-tabulate"}, f"Unexpected val repos: {val_repos}"
assert test_repos == {"requests"}, f"Unexpected test repos: {test_repos}"

assert len(train_repos.intersection(val_repos)) == 0, "Train and Val repositories overlap!"
assert len(train_repos.intersection(test_repos)) == 0, "Train and Test repositories overlap!"
assert len(val_repos.intersection(test_repos)) == 0, "Val and Test repositories overlap!"

print("[OK] Strict dataset boundary assertions verified: 36 Train, 12 Validation, 12 Test (Zero Leakage).")

## 7. Model and Cache Validation
Performs a real frozen LLM smoke test on CUDA GPU:  
1. Loads `Qwen/Qwen2.5-Coder-3B-Instruct` with 4-bit NF4 quantization on CUDA.  
2. Confirms CUDA placement and generates a response.  
3. Verifies token accounting, measures latency and GPU VRAM.  
4. Verifies SQLite operation cache read/write on Google Drive.

In [ ]:
%cd /content/Repo-Guard
import time
import torch
from pathlib import Path
from repoguard.llm import LLMService
from repoguard.agent.cache import OperationCache

print("=== REAL FROZEN LLM SMOKE TEST ===")
MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
print(f"Target Model: {MODEL_ID} (4-bit NF4 on CUDA)")

# 1. Initialize real LLM Service with 4-bit NF4 quantization
llm = LLMService(model_name=MODEL_ID, device="cuda", quantization_bits=4, mock_mode=False)
llm._init_model()

assert llm._model is not None, "FATAL: Model failed to load!"
assert llm._tokenizer is not None, "FATAL: Tokenizer failed to load!"
device_type = next(llm._model.parameters()).device.type
print(f"[OK] Model successfully loaded on device: {device_type}")
assert device_type == "cuda", f"FATAL: Model is not on CUDA, found: {device_type}"

# 2. Test Real Generation & Token Accounting
test_prompt = "Write a Python function named `parse_config` that reads a file path:"
t0 = time.perf_counter()
result = llm.generate(test_prompt, max_new_tokens=32, temperature=0.0)
latency_ms = (time.perf_counter() - t0) * 1000.0

peak_vram_mb = torch.cuda.max_memory_allocated() / (1024**2)
print(f"[OK] Generated Text: {result.text[:80]!r}...")
print(f"[OK] Token Accounting: Prompt={result.prompt_tokens}, Completion={result.completion_tokens}, Total={result.total_tokens}")
print(f"[OK] Latency: {latency_ms:.1f} ms | Peak VRAM: {peak_vram_mb:.1f} MB")

assert result.prompt_tokens > 0, "Prompt tokens must be > 0"
assert result.completion_tokens > 0, "Completion tokens must be > 0"
assert len(result.text.strip()) > 0, "Generated text must not be empty"

# 3. Test Persistent SQLite Operation Cache on Google Drive
cache_file = Path('/content/drive/MyDrive/RepoGuard/data/cache/operations.db')
cache = OperationCache(db_path=cache_file)
cache.set("SMOKE_TEST_OP", "schedule", "commit_abc", {"prompt": test_prompt}, result.model_dump())
assert cache.has("SMOKE_TEST_OP", "schedule", "commit_abc", {"prompt": test_prompt})
cached_out = cache.get("SMOKE_TEST_OP", "schedule", "commit_abc", {"prompt": test_prompt})
assert cached_out["text"] == result.text
print(f"[OK] Operation Cache verified on Google Drive: {cache_file} (hits={cache._hits})")

## 8. RL Environment Validation
Runs unit tests for the RL components and validates the 18-dimensional normalized observation vector and 6 canonical discrete actions.

In [ ]:
%cd /content/Repo-Guard
import sys
import subprocess

# Run all unit tests for Member 3 RL pipeline
subprocess.run([
    sys.executable,
    "-m",
    "pytest",
    "tests/unit/test_agent_cache.py",
    "tests/unit/test_environment.py",
    "tests/unit/test_dqn.py",
    "tests/unit/test_reinforce.py",
    "tests/unit/test_policy_serving.py",
    "-v",
], check=True)

from repoguard.agent.environment import RepoGuardEnv, ACTION_LOOKUP
from repoguard.evaluation.benchmark import load_benchmark
from repoguard.repository.indexer import load_index
from pathlib import Path

cases = load_benchmark(Path("data/questions/phase2_benchmark_v1.jsonl"))
train_cases = [c for c in cases if c.split == 'train']
indexes = {name: load_index(Path("data/indexes") / name) for name in {c.repository_id for c in train_cases}}
env = RepoGuardEnv(benchmark_cases=train_cases, indexes=indexes)

obs, info = env.reset(case_index=0, risk_lambda=1.5)
assert obs.shape == (18,), f"Expected 18-dim observation, got {obs.shape}"
assert len(ACTION_LOOKUP) == 6, f"Expected 6 actions, got {len(ACTION_LOOKUP)}"
print("[OK] RepoGuardEnv observation and action space contracts validated!")


## 9. Offline Transition Collection and Validation
> [!NOTE]
> **Architecture Clarification:**  
> - Offline transitions collected via `scripts/run_episodes.py` are strictly for **environment sanity validation, trace inspection, and operation-cache prewarming (Option C)**.  
> - Offline transitions are **NOT** used to train policies directly; both DQN and REINFORCE train exclusively via active online environment interaction on the `train` split.  
> - Transitions are collected **strictly on the `train` split** (36 cases) to prevent cache contamination of the held-out test split.

In [ ]:
%cd /content/Repo-Guard
import json
import shutil
import sys
import subprocess
from collections import Counter
from pathlib import Path
from repoguard.agent.cache import OperationCache

ep_file = Path('/content/drive/MyDrive/RepoGuard/data/episodes/episode_log.jsonl')
cache_file = Path('/content/drive/MyDrive/RepoGuard/data/cache/operations.db')

# Collect transitions if not already collected
if not ep_file.exists() or ep_file.stat().st_size == 0:
    print("Collecting offline transitions strictly on train split...")
    subprocess.run([
        sys.executable, "scripts/run_episodes.py",
        "--split", "train",
        "--cache-db", str(cache_file),
        "--output", str(ep_file),
    ], check=True)

# Symlink or copy to local data directory
local_ep = Path("data/episodes/episode_log.jsonl")
local_ep.parent.mkdir(parents=True, exist_ok=True)
if not local_ep.exists():
    shutil.copy(ep_file, local_ep)

records = [json.loads(line) for line in ep_file.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"Loaded {len(records)} transitions from {ep_file}")
assert len(records) > 0, "FATAL: Episode log is empty!"

# Strict schema and dimension validation
required_keys = {"case_id", "repository_id", "state", "action", "reward", "next_state", "done", "valid_mask"}
for idx, r in enumerate(records):
    missing = required_keys - set(r.keys())
    assert not missing, f"Record {idx} missing fields: {missing}"
    assert len(r["state"]) == 18, f"Record {idx} state length != 18"
    assert len(r["next_state"]) == 18, f"Record {idx} next_state length != 18"
    assert len(r["valid_mask"]) == 6, f"Record {idx} valid_mask length != 6"

repos_covered = set(r["repository_id"] for r in records)
assert not any(r in {"python-tabulate", "requests"} for r in repos_covered), "Contamination: Val or Test repos in episode log!"
print(f"  [OK] Validated {len(records)} transitions across train repos: {repos_covered}")

cache = OperationCache(db_path=cache_file)
print(f"  [OK] Persistent Operation Cache entries in Drive: {cache.size()}")


## 10. DQN and REINFORCE Checkpoint Smoke Tests
Executes a rigorous two-process recovery verification test for both algorithms:  
1. **DQN Smoke Test:** Process 1 trains for 4 episodes and saves a checkpoint. Process 2 resumes with `--auto-resume` targeting 8 episodes. Confirms restart from episode 5 with complete learning state (weights, optimizer, replay buffer, RNG).
2. **REINFORCE Smoke Test:** Process 1 trains for 3 episodes. Process 2 resumes targeting 6 episodes. Confirms restart from episode 4 with baseline.

In [ ]:
%cd /content/Repo-Guard
import sys
import subprocess

# Run the dedicated two-process recovery test suite
subprocess.run([
    sys.executable,
    "-m",
    "pytest",
    "tests/unit/test_two_process_recovery.py",
    "-v",
], check=True)

print("\n[OK] TWO-PROCESS RECOVERY 100% PASSED FOR BOTH DQN AND REINFORCE!")


## 11. Full DQN Training (2,000 Episodes)

> [!TIP]
> **Overnight Resilience:** `--auto-resume` is active. If Google Colab disconnects, simply re-run this cell; it automatically detects the latest checkpoint on Google Drive and continues seamlessly from where it left off without restarting.

**Hyperparameters:**
- Algorithm: Deep Q-Network (DQN)
- Target Episodes: 2,000
- Seed: 42 (explicit for initial reproducibility)
- Learning Rate: 0.0005
- Batch Size: 32
- Replay Buffer Capacity: 20,000 transitions
- Exploration: $\epsilon: 1.0 \to 0.05$ (decay 0.995)
- Risk tolerance: $\lambda \sim \text{LogUniform}(0.25, 5.0)$
- Dataset Split: Strictly `train` (36 cases)
- Checkpoints: Atomic save to Google Drive every 50 episodes

In [ ]:
%cd /content/Repo-Guard
import sys
import subprocess
from pathlib import Path

cmd = [
    sys.executable, "scripts/train_agent.py",
    "--algo", "dqn",
    "--split", "train",
    "--episodes", "2000",
    "--device", "cuda",
    "--seed", "42",
    "--auto-resume",
    "--learning-rate", "0.0005",
    "--batch-size", "32",
    "--replay-capacity", "20000",
    "--save-interval", "50",
    "--checkpoint-dir", "/content/drive/MyDrive/RepoGuard/checkpoints/dqn",
    "--cache-db", "/content/drive/MyDrive/RepoGuard/data/cache/operations.db",
    "--log-interval", "50",
]
subprocess.run(cmd, check=True)

# Strict post-training assertion
dqn_latest = Path('/content/drive/MyDrive/RepoGuard/checkpoints/dqn/dqn_policy_latest.pt')
assert dqn_latest.exists(), f"FATAL: Expected DQN checkpoint not found at {dqn_latest}"
print("[OK] DQN Training Complete and Verified!")


## 12. Full REINFORCE Training (1,500 Episodes)
Train the REINFORCE baseline over 1,500 episodes with running-mean baseline subtraction.

In [ ]:
%cd /content/Repo-Guard
import sys
import subprocess
from pathlib import Path

cmd = [
    sys.executable, "scripts/train_agent.py",
    "--algo", "reinforce",
    "--split", "train",
    "--episodes", "1500",
    "--device", "cuda",
    "--seed", "42",
    "--auto-resume",
    "--learning-rate", "0.0005",
    "--save-interval", "50",
    "--checkpoint-dir", "/content/drive/MyDrive/RepoGuard/checkpoints/reinforce",
    "--cache-db", "/content/drive/MyDrive/RepoGuard/data/cache/operations.db",
    "--log-interval", "50",
]
subprocess.run(cmd, check=True)

# Strict post-training assertion
rf_latest = Path('/content/drive/MyDrive/RepoGuard/checkpoints/reinforce/reinforce_policy_latest.pt')
assert rf_latest.exists(), f"FATAL: Expected REINFORCE checkpoint not found at {rf_latest}"
print("[OK] REINFORCE Training Complete and Verified!")


## 13. Validation Split Evaluation (Model Selection)
Evaluate both trained policies on the isolated **validation split** (12 cases) across the 6 risk levels $\lambda \in \{0.25, 0.5, 1.0, 2.0, 3.0, 5.0\}$.

In [ ]:
%cd /content/Repo-Guard
import sys
import subprocess
from pathlib import Path

val_dqn_csv = Path('/content/drive/MyDrive/RepoGuard/reports/val_dqn.csv')
val_rf_csv = Path('/content/drive/MyDrive/RepoGuard/reports/val_reinforce.csv')

# 1. Evaluate DQN on Validation Split
subprocess.run([
    sys.executable, "scripts/evaluate_rl.py",
    "--policy", "/content/drive/MyDrive/RepoGuard/checkpoints/dqn/dqn_policy_latest.pt",
    "--split", "validation",
    "--cache-db", "/content/drive/MyDrive/RepoGuard/data/cache/operations.db",
    "--output-csv", str(val_dqn_csv),
], check=True)

# 2. Evaluate REINFORCE on Validation Split
subprocess.run([
    sys.executable, "scripts/evaluate_rl.py",
    "--policy", "/content/drive/MyDrive/RepoGuard/checkpoints/reinforce/reinforce_policy_latest.pt",
    "--split", "validation",
    "--cache-db", "/content/drive/MyDrive/RepoGuard/data/cache/operations.db",
    "--output-csv", str(val_rf_csv),
], check=True)

assert val_dqn_csv.exists(), f"Missing {val_dqn_csv}"
assert val_rf_csv.exists(), f"Missing {val_rf_csv}"
print("[OK] Validation split evaluation complete!")


## 14. Final Held-Out Test Evaluation
Evaluate both policies on the strictly **held-out test split** (12 cases) across all 6 risk levels.  
Note: No policy weights are updated during this final evaluation.

In [ ]:
%cd /content/Repo-Guard
import sys
import subprocess
from pathlib import Path

test_dqn_csv = Path('/content/drive/MyDrive/RepoGuard/reports/test_dqn.csv')
test_rf_csv = Path('/content/drive/MyDrive/RepoGuard/reports/test_reinforce.csv')

# 1. Evaluate DQN on Held-Out Test Split
subprocess.run([
    sys.executable, "scripts/evaluate_rl.py",
    "--policy", "/content/drive/MyDrive/RepoGuard/checkpoints/dqn/dqn_policy_latest.pt",
    "--split", "test",
    "--cache-db", "/content/drive/MyDrive/RepoGuard/data/cache/operations.db",
    "--output-csv", str(test_dqn_csv),
], check=True)

# 2. Evaluate REINFORCE on Held-Out Test Split
subprocess.run([
    sys.executable, "scripts/evaluate_rl.py",
    "--policy", "/content/drive/MyDrive/RepoGuard/checkpoints/reinforce/reinforce_policy_latest.pt",
    "--split", "test",
    "--cache-db", "/content/drive/MyDrive/RepoGuard/data/cache/operations.db",
    "--output-csv", str(test_rf_csv),
], check=True)

assert test_dqn_csv.exists(), f"Missing {test_dqn_csv}"
assert test_rf_csv.exists(), f"Missing {test_rf_csv}"
print("[OK] Final held-out test evaluation complete!")


## 15. Risk-Curve Plots and Comparative Analysis
Load final test CSVs and plot the 3 core risk trade-off curves:  
1. **Coverage vs. $\lambda$**: Selective answering under high risk.  
2. **Hallucination Rate vs. $\lambda$**: Hallucination suppression.  
3. **Trap Abstention vs. $\lambda$**: Proactive abstention on unanswerable questions.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

dqn_test = pd.read_csv('/content/drive/MyDrive/RepoGuard/reports/test_dqn.csv')
rf_test = pd.read_csv('/content/drive/MyDrive/RepoGuard/reports/test_reinforce.csv')

print("=== HELD-OUT TEST RESULTS: DQN ===")
display(dqn_test)

print("\n=== HELD-OUT TEST RESULTS: REINFORCE ===")
display(rf_test)

def get_col(df, *names):
    for n in names:
        if n in df.columns:
            return df[n]
    raise KeyError(f"None of {names} found in {df.columns}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

lam_dqn = get_col(dqn_test, 'lambda', 'Lambda')
lam_rf = get_col(rf_test, 'lambda', 'Lambda')

# 1. Coverage
axes[0].plot(lam_dqn, get_col(dqn_test, 'coverage_pct', 'Coverage (%)'), 'o-', label='DQN (Ours)', color='royalblue', linewidth=2)
axes[0].plot(lam_rf, get_col(rf_test, 'coverage_pct', 'Coverage (%)'), 's--', label='REINFORCE', color='darkorange', linewidth=2)
axes[0].set_xlabel('Risk Penalty (lambda)', fontsize=11)
axes[0].set_ylabel('Coverage (%)', fontsize=11)
axes[0].set_title('Answer Coverage vs. Risk Tolerance', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# 2. Hallucination Rate
axes[1].plot(lam_dqn, get_col(dqn_test, 'hallucination_rate_pct', 'Hallucination Rate (%)'), 'o-', label='DQN (Ours)', color='crimson', linewidth=2)
axes[1].plot(lam_rf, get_col(rf_test, 'hallucination_rate_pct', 'Hallucination Rate (%)'), 's--', label='REINFORCE', color='purple', linewidth=2)
axes[1].set_xlabel('Risk Penalty (lambda)', fontsize=11)
axes[1].set_ylabel('Hallucination Rate (%)', fontsize=11)
axes[1].set_title('Hallucination Rate vs. Risk Tolerance', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# 3. Correct Abstention on Traps
axes[2].plot(lam_dqn, get_col(dqn_test, 'correct_abstention_pct', 'Correct Abstention (%)'), 'o-', label='DQN (Ours)', color='forestgreen', linewidth=2)
axes[2].plot(lam_rf, get_col(rf_test, 'correct_abstention_pct', 'Correct Abstention (%)'), 's--', label='REINFORCE', color='teal', linewidth=2)
axes[2].set_xlabel('Risk Penalty (lambda)', fontsize=11)
axes[2].set_ylabel('Trap Abstention (%)', fontsize=11)
axes[2].set_title('Correct Trap Abstention vs. Risk Tolerance', fontsize=12, fontweight='bold')
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
chart_path = Path('/content/drive/MyDrive/RepoGuard/reports/test_risk_curves.png')
chart_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(str(chart_path), dpi=300)
plt.show()
assert chart_path.exists(), f"Figure not saved at {chart_path}"
print(f"Risk curves figure successfully saved to: {chart_path}")


## 16. Experiment Manifest and Artifact Summary
Records actual measured values from training execution:  
- Project Git commit hash and repository commit hashes  
- Model IDs and quantization configuration  
- Benchmark split counts and hyperparameters  
- Measured GPU hardware, total VRAM, and runtime versions  
- Actual training episodes and duration  
- Checkpoint and report paths on Google Drive

In [ ]:
%cd /content/Repo-Guard
import sys
import subprocess
from pathlib import Path

val_dqn_csv = Path('/content/drive/MyDrive/RepoGuard/reports/val_dqn.csv')
val_rf_csv = Path('/content/drive/MyDrive/RepoGuard/reports/val_reinforce.csv')

# 1. Evaluate DQN on Validation Split
subprocess.run([
    sys.executable, "scripts/evaluate_rl.py",
    "--policy", "/content/drive/MyDrive/RepoGuard/checkpoints/dqn/dqn_policy_latest.pt",
    "--split", "validation",
    "--cache-db", "/content/drive/MyDrive/RepoGuard/data/cache/operations.db",
    "--output-csv", str(val_dqn_csv),
], check=True)

# 2. Evaluate REINFORCE on Validation Split
subprocess.run([
    sys.executable, "scripts/evaluate_rl.py",
    "--policy", "/content/drive/MyDrive/RepoGuard/checkpoints/reinforce/reinforce_policy_latest.pt",
    "--split", "validation",
    "--cache-db", "/content/drive/MyDrive/RepoGuard/data/cache/operations.db",
    "--output-csv", str(val_rf_csv),
], check=True)

assert val_dqn_csv.exists(), f"Missing {val_dqn_csv}"
assert val_rf_csv.exists(), f"Missing {val_rf_csv}"
print("[OK] Validation split evaluation complete!")
